# Data Science 1 enrichment · Regression in more depth

**Lesson:** Data Science 1, Day 1 · **Time:** about 45 minutes, optional · **ISLP:** 3.1.2, 3.2.2, 4.3.1–4.3.4 · **Data:** CADets release 1

**What you will do**
- Part A, linear regression: why the scale of a predictor changes its coefficient, and how the standard error, t-statistic and p-value of a coefficient are calculated
- Part B, logistic regression: the logit function, moving between a coefficient and an odds ratio, and how predicted risks lead to the ROC curve and AUROC of Data Science 3

Run each cell in order. Cells marked **Try it** invite you to change a value and run again.

In [ ]:
# Setup: load the CADets course dataset. Run this cell first.
# RELEASE: 1 from Day 1, 2 from Day 2, 3 from Day 5. Use the release participants have on your lesson day.
RELEASE = 1

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = {1: "cadets_admissions_release1_core.csv",
         2: "cadets_admissions_release2_intervention.csv",
         3: "cadets_admissions_release3_full.csv"}
DATA_URL = "https://raw.githubusercontent.com/Rotem-Gershon/cadets-notebooks/main/data/"

def load_cadets(release=RELEASE):
    name = FILES[release]
    if os.path.exists(name):  # a copy uploaded to this session
        return pd.read_csv(name)
    try:  # the course repository, from the release day on
        return pd.read_csv(DATA_URL + name)
    except Exception:
        pass
    try:  # before the release day: upload the file from the course website's dataset page
        from google.colab import files
        print(f"Release {release} is not published yet. Upload {name} from the course website's dataset page.")
        files.upload()
        return pd.read_csv(name)
    except ImportError:
        raise FileNotFoundError(f"Put {name} in the same folder as this notebook.")

df = load_cadets()
print(f"Release {RELEASE}: {df.shape[0]:,} admissions, {df.shape[1]} columns")

In [ ]:
import statsmodels.formula.api as smf
from scipy import stats

# Part A · Linear regression

## A1 · Scaling: coefficients depend on units

Fit length of stay on four predictors measured in very different units.

In [ ]:
predictors = ["age", "resp_rate", "creatinine", "spo2"]
formula = "los_days ~ " + " + ".join(predictors)
model_raw = smf.ols(formula, data=df).fit()
print(model_raw.params.round(3))
df[predictors].describe().loc[["mean", "std"]].round(2)

Creatinine has the largest coefficient, but one mg/dL of creatinine is a big change, while one year of age is a small one. The size of a raw coefficient tells you about the unit, not about how important the predictor is.

The same model with creatinine in µmol/L (× 88.4) gives a coefficient 88.4 times smaller, and nothing else changes:

In [ ]:
df["creatinine_umol"] = df["creatinine"] * 88.4
model_umol = smf.ols("los_days ~ age + resp_rate + creatinine_umol + spo2", data=df).fit()
print("Creatinine per mg/dL: ", round(model_raw.params["creatinine"], 3))
print("Creatinine per µmol/L:", round(model_umol.params["creatinine_umol"], 4))
print("R² both models:       ", round(model_raw.rsquared, 4), round(model_umol.rsquared, 4))

### Standardized predictors

To compare predictors on one scale, **standardize** them: subtract the mean and divide by the standard deviation. A coefficient then means *days per one standard deviation* of the predictor.

In [ ]:
df_std = df.copy()
for col in predictors:
    df_std[col] = (df[col] - df[col].mean()) / df[col].std()

model_std = smf.ols(formula, data=df_std).fit()
compare = pd.DataFrame({"raw β": model_raw.params, "standardized β": model_std.params,
                        "p raw": model_raw.pvalues, "p standardized": model_std.pvalues})
compare.drop(index="Intercept").round(4)

What changed and what did not:
- The coefficients changed, and now they can be compared: respiratory rate and SpO₂ move length of stay most per standard deviation.
- The p-values and R² are identical. Scaling changes units, not the evidence.
- The intercept is now the predicted stay for a patient at the mean of every predictor, which is easier to read than a stay at age 0.

Scaling matters even more for the models of Data Science 2 onwards: methods fitted by gradient descent, penalized regression and neural networks all work better, or only work, when predictors are on similar scales.

## A2 · How is a coefficient's significance calculated?

`summary()` prints a standard error, a t-statistic and a p-value for each coefficient. Here they are by hand for the model of the lesson.

In [ ]:
model = smf.ols("los_days ~ age + heart_failure + ckd + resp_rate", data=df).fit()
X = np.column_stack([np.ones(len(df)), df[["age", "heart_failure", "ckd", "resp_rate"]]])
y = df["los_days"].values

beta = np.linalg.solve(X.T @ X, X.T @ y)          # least squares
residuals = y - X @ beta
n, p = X.shape
sigma2 = residuals @ residuals / (n - p)          # residual variance
se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X.T @ X)))

1. **β**: the least-squares solution.
2. **Residual variance σ²**: how far patients fall from the fitted line, divided by n − p degrees of freedom.
3. **Standard error (SE)**: how much β would vary from sample to sample. It is small when the residuals are small, n is large and the predictor varies a lot.
4. **t = β / SE**: how many standard errors β is away from 0.
5. **p-value**: the chance of a |t| this large if the true coefficient were 0, from the t distribution with n − p degrees of freedom.
6. **95% CI**: β ± t₀.₉₇₅ × SE.

In [ ]:
t = beta / se
pval = 2 * stats.t.sf(np.abs(t), df=n - p)
t_crit = stats.t.ppf(0.975, df=n - p)
by_hand = pd.DataFrame({"β": beta, "SE": se, "t": t, "p": pval,
                        "CI low": beta - t_crit * se, "CI high": beta + t_crit * se},
                       index=model.params.index)
by_hand.round(4)

In [ ]:
model.summary().tables[1]

The two tables match. Two things to keep in mind:
- A p-value depends on n. With 2,000 admissions a tiny effect can be significant; with 50 a large one may not be. Read the coefficient and its CI, not only p.
- A significant coefficient is an association adjusted for the other predictors in the model. Whether it is causal is a question for Causal Inference 1–3.

**Try it:** fit the same model on `df.sample(200, random_state=1)` and compare the SEs and p-values with the full data.

In [ ]:
small = smf.ols("los_days ~ age + heart_failure + ckd + resp_rate",
                data=df.sample(200, random_state=1)).fit()
pd.DataFrame({"SE n=2000": model.bse, "SE n=200": small.bse,
              "p n=2000": model.pvalues, "p n=200": small.pvalues}).round(4)

# Part B · Logistic regression

## B1 · The logit function

A probability lives between 0 and 1, but a sum of predictors can be any number. The **logit** bridges the two:

logit(p) = log( p / (1 − p) ) = log-odds

Logistic regression sets the log-odds equal to β₀ + β₁x₁ + … and the **sigmoid** (inverse logit) brings it back to a probability: p = 1 / (1 + e^(−log-odds)).

In [ ]:
p = np.array([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])
pd.DataFrame({"probability": p, "odds": p / (1 - p), "log-odds": np.log(p / (1 - p))}).round(3)

In [ ]:
z = np.linspace(-6, 6, 200)
plt.plot(z, 1 / (1 + np.exp(-z)), color="black")
plt.axhline(0.5, ls=":", color="grey"); plt.axvline(0, ls=":", color="grey")
plt.xlabel("Log-odds (β₀ + β₁x₁ + …)"); plt.ylabel("Probability")
plt.title("The sigmoid: inverse of the logit")
plt.show()

- Probability 0.5 is log-odds 0. Below 0.5 the log-odds are negative.
- Near 0 and 1 the curve flattens: the same change in log-odds moves the probability less.
- For a rare outcome odds ≈ probability (0.01 vs 0.0101), so an OR is close to a risk ratio. In-hospital death here is 9%, where that approximation starts to fail.

Check it on a patient: the model's linear predictor, passed through the sigmoid, is exactly `predict`.

In [ ]:
model_death = smf.logit("in_hospital_death ~ heart_failure + age + ckd + resp_rate + spo2",
                        data=df).fit(disp=0)
patient = df.iloc[[0]]
log_odds = model_death.predict(patient, which="linear").iloc[0]
print("Log-odds:           ", round(log_odds, 3))
print("Sigmoid of log-odds:", round(1 / (1 + np.exp(-log_odds)), 4))
print("model.predict:      ", round(model_death.predict(patient).iloc[0], 4))

## B2 · From coefficient to odds ratio and back

Because the model is linear on the log-odds scale, a one-unit increase in x adds β to the log-odds, which multiplies the odds by e^β. So:

| You have | You want | Do |
|---|---|---|
| β | OR per 1 unit | exp(β) |
| β | OR per k units | exp(k × β) |
| β | OR in the other direction | exp(−β) = 1 / OR |
| OR | β | log(OR) |
| CI of β | CI of OR | exp of each limit |

In [ ]:
b = model_death.params
ci = model_death.conf_int()
print("Age β:                ", round(b["age"], 4))
print("OR per year:          ", round(np.exp(b["age"]), 3))
print("OR per 10 years:      ", round(np.exp(10 * b["age"]), 2))
print("SpO₂ OR per 1% higher:", round(np.exp(b["spo2"]), 3))
print("SpO₂ OR per 1% lower: ", round(np.exp(-b["spo2"]), 3))
print("Heart failure 95% CI of OR:", np.exp(ci.loc["heart_failure"]).round(2).tolist())

Going back from a published OR to β is useful when you want to plug a reported effect into your own risk calculation. Suppose a paper reports OR 1.54 for heart failure:

In [ ]:
reported_or = 1.54
beta_hf = np.log(reported_or)
print("β = log(OR) =", round(beta_hf, 3))

# For a patient with baseline risk 10%, what is the risk with heart failure?
baseline_risk = 0.10
new_log_odds = np.log(baseline_risk / (1 - baseline_risk)) + beta_hf
print("Risk with heart failure:", round(1 / (1 + np.exp(-new_log_odds)), 3))

An OR of 1.54 turns a 10% risk into 14.6%, not 15.4%: the OR multiplies odds, not risks.

**Try it:** change `baseline_risk` to 0.01 and to 0.50. Where is the OR closest to a risk ratio?

## B3 · From predicted risk to AUROC

A logistic regression's product is a predicted risk for each patient. How good is that risk at telling apart patients who died from those who survived? Data Science 3 answers this in full; here is how the pieces connect.

**Note:** everything below uses the same patients the model was fitted on. That flatters the model. Data Science 2 explains why and holds out a test set.

In [ ]:
df["predicted_risk"] = model_death.predict(df)
died = df["in_hospital_death"] == 1
bins = np.linspace(0, 0.8, 33)
plt.hist(df.loc[~died, "predicted_risk"], bins=bins, alpha=0.6, density=True, label="Survived")
plt.hist(df.loc[died, "predicted_risk"], bins=bins, alpha=0.6, density=True, label="Died")
plt.xlabel("Predicted risk of in-hospital death"); plt.ylabel("Density"); plt.legend()
plt.show()

The two groups overlap, but patients who died tend to have higher predicted risks. To act on the risk, pick a **threshold**: flag everyone above it. Each threshold gives a **sensitivity** (share of deaths flagged) and a **specificity** (share of survivors not flagged).

In [ ]:
def sens_spec(threshold):
    flagged = df["predicted_risk"] >= threshold
    return (flagged & died).sum() / died.sum(), (~flagged & ~died).sum() / (~died).sum()

for threshold in [0.05, 0.10, 0.20, 0.30]:
    s, sp = sens_spec(threshold)
    print(f"Threshold {threshold:.2f}: sensitivity {s:.2f}, specificity {sp:.2f}")

Lower thresholds catch more deaths and raise more false alarms. Plot sensitivity against 1 − specificity for every threshold and you get the **ROC curve**. The area under it is the **AUROC**.

In [ ]:
thresholds = np.unique(df["predicted_risk"])[::-1]
points = np.array([sens_spec(t) for t in thresholds])
tpr, fpr = np.r_[0, points[:, 0]], np.r_[0, 1 - points[:, 1]]
auc = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)   # area by trapezoids

plt.plot(fpr, tpr, color="black"); plt.plot([0, 1], [0, 1], ls=":", color="grey")
plt.xlabel("1 − specificity"); plt.ylabel("Sensitivity")
plt.title(f"ROC curve, AUROC = {auc:.3f}")
plt.show()

AUROC has a direct meaning: pick one patient who died and one who survived at random. AUROC is the probability that the model gave the patient who died the higher risk. Check it by comparing every such pair:

In [ ]:
r_died = df.loc[died, "predicted_risk"].values
r_surv = df.loc[~died, "predicted_risk"].values
pairs = r_died[:, None] > r_surv[None, :]
ties = r_died[:, None] == r_surv[None, :]
print("Share of pairs ranked correctly:", round(pairs.mean() + 0.5 * ties.mean(), 3))
print("AUROC from the curve:           ", round(auc, 3))

Three things follow, and Data Science 3 builds on them:
- AUROC depends only on the **ranking** of patients, not on the risk values. Multiply every risk by 2 and AUROC is unchanged, even though the risks are now wrong. Whether the risks are right is **calibration**.
- AUROC does not pick a threshold. The threshold is a product decision: how many alerts per 100 admissions the ward can handle, and how many deaths it is willing to miss.
- The odds ratios of Part B2 and the AUROC answer different questions. A predictor with a strong OR can still add little to AUROC if few patients have it.

**Try it:** refit the model with heart failure only and recompute the AUROC. How much do age, CKD, respiratory rate and SpO₂ add?

## Key points

- Raw coefficients reflect units. Standardize predictors to compare them; p-values and R² do not change.
- A coefficient's SE comes from the residual variance and the spread of the predictor; t = β / SE gives the p-value. p depends on n.
- The logit maps a probability to log-odds; the sigmoid maps it back. exp(β) is an OR, exp(k × β) is the OR per k units, log(OR) is β.
- Thresholding the predicted risk gives sensitivity and specificity; all thresholds together give the ROC curve, and AUROC is the chance a patient who died is ranked above one who survived.

## Further reading

- ISLP, sections 3.1.2 (accuracy of the coefficient estimates), 3.2.2, 4.3.1–4.3.4 and 4.4.2 (the ROC curve). Free at [statlearning.com](https://www.statlearning.com).